# CNN: VGG16 Workshop Tutorial

This notebook shows how to load a VGG16 benchmark network in StarV, build the corresponding input Star set, and read the VNNLIB property used for verification.

## 1. Setup

Import the StarV utilities used in the workshop and resolve paths relative to the local StarV package. This keeps the notebook working even when Jupyter is launched from a different folder.


In [1]:
# Numerical arrays are used for bounds and parsed VNNLIB data.
import numpy as np
import StarV

# Star is the symbolic set representation used by StarV reachability routines.
from StarV.set.star import Star

# load_ACASXU loads the curated ACASXU benchmark from StarV MAT files.
# load_neural_network demonstrates loading a network file such as ONNX.
from StarV.util.load import load_ACASXU, load_neural_network

# VNNLIB helpers read benchmark input boxes and output safety constraints.
from StarV.util.vnnlib import read_vnnlib_simple, get_num_inputs_outputs
from StarV.util.lp_solver import sample

# Verification helpers run approximate reachability.
from StarV.verifier.certifier import reachBFS

from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torchvision.models as models

# repo_root points at the repository root that contains the StarV package folder.
repo_root = Path(StarV.__file__).resolve().parents[1]


def as_list(reach_set):
    """Normalize a Star or list of Stars for plotting/counting."""
    return reach_set if isinstance(reach_set, list) else [reach_set]

## 2. Load The VGG16 Network and VNNLIB Property

This section mirrors a standard verification benchmark layout: a VGG16 network plus a VNNLIB property file. The VNNLIB parser provides the input box and output constraints

In [2]:
# Load the VGG16 model with pre-trained weights
model = models.vgg16(weights='DEFAULT')

# Set the model to evaluation mode for inference
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [4]:
net_type = 'VGG16_CNN'
net = load_neural_network(model, net_type=net_type)
print(net)

torch.nn.modules.container.Sequential layer is neglected in the analysis
torch.nn.modules.container.Sequential layer is neglected in the analysis

=============NETWORK===============
Network type: VGG16_CNN
Input Dimension: 3
Output Dimension: 1000
Number of Layers: 37
Layer types:
Layer 0: <class 'StarV.layer.Conv2DLayer.Conv2DLayer'> (3, 64, kernel_size = (3, 3), stride = [1 1], padding = [1 1 1 1], dilation = [1 1], dtype=float64)
Layer 1: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 2: <class 'StarV.layer.Conv2DLayer.Conv2DLayer'> (64, 64, kernel_size = (3, 3), stride = [1 1], padding = [1 1 1 1], dilation = [1 1], dtype=float64)
Layer 3: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 4: <class 'StarV.layer.MaxPool2DLayer.MaxPool2DLayer'> (kernel_size = [2 2], stride = [2 2], padding = [0 0])
Layer 5: <class 'StarV.layer.Conv2DLayer.Conv2DLayer'> (64, 128, kernel_size = (3, 3), stride = [1 1], padding = [1 1 1 1], dilation = [1 1], dtype=float64)
Layer 6: <class 'StarV.layer.Re